In [ ]:
import os
import glob
import warnings

import rasterio
import numpy as np
from matplotlib import pyplot as plt

import eos.sar 
from eos.sar.roi import Roi
import eos.products.sentinel1 as s1
from eos.products.sentinel1 import mosaic

## The data
The data that will be used for this experiment is a couple of S1A acquisitions

Reference: `S1A_IW_SLC__1SDV_20211229T231926_20211229T231953_041230_04E66A_3DBE`


Secondary: `S1A_IW_SLC__1SDV_20220110T231926_20220110T231953_041405_04EC57_103E`

They span an earthquake taking place at January 7 2022: M 6.6 - 113 km SW of Jinchang, China:
https://sarviews-hazards.alaska.edu/Event/e2dfcb22-e1a4-43d8-a17e-c6b175849463

To download the data, we use the script provided by ASF (after slight modifications).
In a shell, from the directory `eos-sar`, run: 

    python tools/download_from_asf.py https://s1qc.asf.alaska.edu/aux_resorb/S1A_OPER_AUX_RESORB_OPOD_20211230T024411_V20211229T224022_20211230T015752.EOF usage/tutorial/data/orb
    python tools/download_from_asf.py https://s1qc.asf.alaska.edu/aux_resorb/S1A_OPER_AUX_RESORB_OPOD_20220111T024731_V20220110T224022_20220111T015752.EOF usage/tutorial/data/orb
    python tools/download_from_asf.py https://datapool.asf.alaska.edu/SLC/SA/S1A_IW_SLC__1SDV_20211229T231926_20211229T231953_041230_04E66A_3DBE.zip usage/tutorial/data/safes --unzip
    python tools/download_from_asf.py https://datapool.asf.alaska.edu/SLC/SA/S1A_IW_SLC__1SDV_20220110T231926_20220110T231953_041405_04EC57_103E.zip usage/tutorial/data/safes --unzip
   
   
The two products will be downloaded and unzipped in the directory. The two corresponding orbits will also be downloaded.

In [ ]:
assert os.path.exists('./tutorial/data/safes/S1A_IW_SLC__1SDV_20211229T231926_20211229T231953_041230_04E66A_3DBE.SAFE')
assert os.path.exists('./tutorial/data/safes/S1A_IW_SLC__1SDV_20220110T231926_20220110T231953_041405_04EC57_103E.SAFE')
assert os.path.exists('./tutorial/data/orb/S1A_OPER_AUX_RESORB_OPOD_20211230T024411_V20211229T224022_20211230T015752.EOF')
assert os.path.exists('./tutorial/data/orb/S1A_OPER_AUX_RESORB_OPOD_20220111T024731_V20220110T224022_20220111T015752.EOF')

In [ ]:
# Input/Output helper functions
def pid_from_safe_dir(safe_dir):
    return os.path.splitext(os.path.basename(safe_dir))[0]

def pid_to_date(product_id):
    return product_id.split('_')[5][:8]

def write_array(array, path):
    height, width = array.shape
    with warnings.catch_warnings():
        warnings.filterwarnings("ignore", category=FutureWarning)
        warnings.filterwarnings("ignore", category=rasterio.errors.NotGeoreferencedWarning)
        profile = dict(count=1, width=width, height=height, dtype=array.dtype)
        with rasterio.open(path, "w", **profile) as f:
            f.write(array, 1)

In [ ]:
workdir = "./tutorial"

out_path = os.path.join(workdir, "results")
if not os.path.exists(out_path):
    os.makedirs(out_path)

## Load metadata

### Sentinel1ProductInfo

Sentinel-1 images are distributed in \"products\". As mentionned previously, we downloaded two products, one for 2021-12-29 and one for 2022-01-10. When the file is unzipped, a directory named \<product_id\>.SAFE is created. In this directory are located the images in the tiff format and the necessary metadata in the xml format.
To access easily a product content, an object of type `Sentinel1ProductInfo` can be used. 

In [ ]:
safe_dirs = glob.glob(os.path.join(workdir, "data", "safes", "*.SAFE"))
safe_dirs = sorted(safe_dirs, key=lambda x: pid_to_date(pid_from_safe_dir(x)))

products = [s1.product.SafeSentinel1ProductInfo(safe_dir) for safe_dir in safe_dirs]

In [ ]:
from pprint import pprint
# A product info object can be used to get a reader on an sentinel-1 tiff
im_reader = products[0].get_image_reader("iw1", "vv")
pprint( im_reader.read(1, window=((2000,2010),(2000,2010))) )

# It can also be used to get the content of the xml metadata files 
xml_content = products[0].get_xml_annotation("iw1", "vv")
pprint(xml_content[:1000])

### Precise orbit

In [ ]:
def get_orbit_provider(local_dir, orbit_type: str):
    
    # by default, xml orbits will be used, no orbit provider function
    orbit_provider = None
    
    # if orbit_type True, or specified as orbpoe (precise orbits) / orbres (restitued orbits)
    if orbit_type:
        
        if orbit_type == True:
            # Take the best quality available for the product
            orbit_type = None
        
        # define the function that acts on the metadata of a product
        def orbit_provider(pid, bursts):
            s1.orbits.update_statevectors_using_local_folder(local_dir, pid, bursts, force_type=orbit_type)
            
    return orbit_provider

In [ ]:
local_dir = os.path.join(workdir, "data", "orb")
orbit_provider = get_orbit_provider(local_dir, orbit_type=True)

# get the metas, with orbits from the local directory, for the first date
bursts_metas = mosaic.get_bursts([products[0]], "iw2", "vv", orbit_provider)[0]

pprint(bursts_metas)

### S1Assembler

This object holds the metadata of all the bursts of all the products at a specific date. It will enable easy instantiation of other objects as we will see later on.

In [ ]:
# give it the list of products of a date, restrict to a swath optionnally
asm = mosaic.S1Assembler.from_products([products[0]], orbit_provider=orbit_provider, swaths=("iw2",))

## Digital Elevation Model (DEM)

Many processings require access to a DEM. An interface to define how dem data should be accessed has been implemented in the form of a DEMSource object. Any object of this class should be able to query a dem at a certain (lon, lat) location, or to crop a dem from the given bounds.

Before running the cell below, ensure you have installed [srtm4](https://github.com/centreborelli/srtm4) (open source) or multidem (Kayrros package). Otherwise, you would need to implement your own source by inheriting from eos.dem.DEMSource.

In [ ]:
# If you wish to localize a point without giving the altitude
# you can use a eos.dem.DEMSource to get a query function to a dem
dem_source = eos.dem.get_any_source()

lon, lat = 20, 40
alt = dem_source.elevation(lon, lat)
print(alt, "m")

lon_min, lat_min, lon_max, lat_max = 20,10,20.03,10.03
raster, transform, crs = dem_source.crop((lon_min, lat_min, lon_max, lat_max))
print(raster.shape, crs)

## Model tutorial

The metadata was previously read and stored in dictionnaries. From the metadata, it is possible to instantiate a eos.sar.model.SensorModel (this is the base class) object.

A "SensorModel" is an object that mainly has the responsability to perform geolocation operations.

In [ ]:
# Let us test a "swath" model
# A swath is defined as the minimal mosaic (minimal in height, width)
# containing the given bursts

# all bursts in the first product are used to get a swath model
test_swath_model = s1.proj_model.swath_model_from_bursts_meta(bursts_metas)

# You can also create a burst model
# here, we take the first burst of the first product
test_burst_model = s1.proj_model.burst_model_from_burst_meta(bursts_metas[0])

# Alternatively, get the projection model from the assembler
alt_swath_model = asm.get_proj_model()

In [ ]:
# Here we do a plot to show what a swath is 

import matplotlib.patches as patches

fig, ax = plt.subplots(figsize=(10,10))

def add_rect(col, row, w, h, color='b'): 
    rect = patches.Rectangle((col, row), w, h, 
                         linewidth=1, edgecolor=color, facecolor='none')
    # Add the patch to the Axes
    ax.add_patch(rect)

w, h = test_swath_model.w, test_swath_model.h
ax.set_xlim(0, w)
ax.set_ylim(h, 0)

for bid in range(len(bursts_metas)):
    col, row = test_swath_model.burst_orig_in_swath(bid)
    h, w = test_swath_model.bursts_rois[bid].get_shape()
    if bid % 2: 
        color = 'r'
    else: 
        color = 'b'

     # Create a Rectangle patch on the swath
    add_rect(col, row, w, h, color)

plt.show()

In the plot, you can see that the swath is composed of 9 bursts. The bursts' color alternates between red and blue. You can see that the bursts overlap (at the end of a burst and the start of the next one). The bursts may also be slightly shifted in the column direction. The swath limits are defined as the minimal limits containing all the bursts.

In [ ]:
# Let us test some geolocation functions
# a set of points in the image, in this case in the swath, can be geolocated
rows = np.round(np.random.rand(5) * (test_swath_model.h - 1))
cols = np.round(np.random.rand(5) * (test_swath_model.w - 1))
# for localization, you need to give the altitude to get the 3D point
# because the 3D point falling in a pixel is ambiguous without the altitude

# for example, finding the 3D points on the ellipsoid for a set of image pixels
alts = np.zeros(5)
lons, lats, alts = test_swath_model.localization(rows, cols, alts)

# Alternatively, the location of a set of 3D points in the image can be found
rows_pred, cols_pred, incidence_pred = test_swath_model.projection(lons, lats, alts)

np.testing.assert_allclose(rows_pred, rows, atol=1e-2)
np.testing.assert_allclose(cols_pred, cols, atol=1e-2)

# If you wish to localize a point without giving the altitude
# you can use a eos.dem.DEMSource to get a query function to a dem
dem_source = eos.dem.get_any_source()

# Then, the first 3D point lying on the dem that corresponds to the pixel is returned
# by this function call
lons, lats, alts, masks = test_swath_model.localize_without_alt(rows, cols, elev=dem_source.elevation)

# In the same logic, the four image corners can be localized to get an estimation of the footprint of the image
approx_geom, alts, masks = test_swath_model.get_approx_geom()

## Region of interest

eos was designed to be able to work on sub-regions within an image. The region can be defined in geographic coordinates as well as coordinates within the image.

In [ ]:
def get_roi(input_geometry, proj_model, dem_source=None):
    # find the region in the image to be studied
    # To do so, we use the model to geolocate (project) the 3D points into the image

    x = [pt[0] for pt in input_geometry]
    y = [pt[1] for pt in input_geometry]

    # we need to find the alt to have the 3D coordinates
    # we use an interface to access and interpolate dem data
    # If you have srtm4 installed, this will be an SRTM90 DEM interface
    if dem_source is None:
        dem_source = eos.dem.get_any_source()

    alt = dem_source.elevation(x, y)

    # projection
    rows, cols, incidences = proj_model.projection(x, y, alt)

    roi_in_img = Roi.from_bounds_tuple(Roi.points_to_bbox(rows, cols))

    return roi_in_img

# We can start defining the area of interest

# you can define it as a list of coordinates (lon, lat)
input_geometry = [
    (101.63079789214869, 38.119580588719934),  # (lon, lat)
    (100.95483833640877, 38.21396403905346),
    (100.7799350202732, 37.47156838966397),
    (101.45975167385876, 37.37555836504315)
]


# In this case, the region of interest was defined in (lon, lat) coordinates
# Then it was transformed to image coordinates
roi_in_img = get_roi(input_geometry, test_swath_model)
pprint(roi_in_img)

# Another way is for example to want to get the whole swath
# if you set primary_swath_roi to None, the whole swath will be considered
# Change the comment here to see what happens during the processing!
# roi_in_img = None

# Also, you can set the roi yourself, but it is hard to know how to set it correctly
# if you don't have a mosaic tif image. For example:
# roi_in_img = Roi(col=48, row=569, w=16221, h=6022)

## Processing flags

In [ ]:
swaths = ("iw2",)
polarization = "vv"

get_complex = True  # whether to work with complex images, or just amplitudes
reramp = True # whether to reramp the complex image 
# (Reramping is needed for interferometry/coherence,
# If no Interferometric application is intended, then
# should be set to False if we wish to do further resampling on the complex image)

calibration = "sigma" # (either None, "sigma", "gamma", "beta")

## Putting everything together
###  Debursting

As defined previously, the region of interest intersects many bursts. To get a crop only on our roi, \"debursting\" should be performed. This operation has many steps that should be applied: 
 - A digital elevation model is used to get 3D points on the region of interest
 - The points are projected in the different bursts of the reference image
 - The points are projected again in the different bursts, with some corrections on the final px position enabled. The corrections are related to propagation effects (Atmospheric path delay for ex.) and to some Sentinel-1 specific distortions ( intra-pulse, bistatic...)
 - One resampling matrix for each burst is estimated 
 - The data intersecting the aoi is identified, read, calibrated, resampled, stitched

Resampling occurs for the primary image in case any correction is enabled. Resampling always occurs for the secondary image to perform the registration.

All theses operations are hidden from the user in the `S1AssemblyCropper` object.

In [ ]:
local_dir = os.path.join(workdir, "data", "orb")
# best orbits from local dir
orbit_provider = get_orbit_provider(local_dir, orbit_type=True)

# Take assembler for products of primary date
asm = mosaic.S1Assembler.from_products([products[0]], orbit_provider=orbit_provider, swaths=swaths)

# get dem interface
dem = eos.dem.get_any_source()

# get projection model 
proj_model = asm.get_proj_model()

# Project geometry into the image (here defined by iw2 swath, from Assembler instantiation)
roi_in_primary = get_roi(input_geometry, proj_model, dem)

# The assembler can create an S1AssemblyCropper object, that is set up on our roi
cropper = asm.get_cropper(roi_in_primary)

print(type(cropper))

In [ ]:
# The cropper can now simply crop an image on our roi, performing all complex steps required 

# For the reference image
primary_debursted_crop = cropper.crop([products[0]], pol=polarization, orbit_provider=orbit_provider,
                get_complex=get_complex, reramp=reramp, dem=dem, calibration=calibration)

# For a secondary image
secondary_debursted_crop = cropper.crop([products[1]], pol=polarization, orbit_provider=orbit_provider,
                get_complex=get_complex, reramp=reramp, dem=dem, calibration=calibration)

## Cropping Results

In [ ]:
# Write the results
write_array(primary_debursted_crop, os.path.join(out_path, "primary_crop.tif"))
write_array(secondary_debursted_crop, os.path.join(out_path, "secondary_crop.tif"))

In [ ]:
def multilook(u, filter_size=(5,21)):
    import scipy.ndimage as ndimage
    mask = np.isnan(u)
    mlooked = u.copy()
    mlooked[mask] = 0
    mlooked = ndimage.uniform_filter(mlooked, size=filter_size, mode="nearest")[::filter_size[0], ::filter_size[1]]
    return mlooked

def display_amp(cmplx_img, filter_size=(5,21)):
    intensity = np.abs(cmplx_img)**2
    amp = np.sqrt(multilook(intensity, filter_size))
    plt.figure(figsize=(12,10))
    vmin, vmax = np.nanpercentile(amp, [10, 90])
    plt.imshow(amp, cmap='gray', vmin=vmin, vmax=vmax)
    plt.colorbar()
    plt.show()


In [ ]:
display_amp(primary_debursted_crop)
display_amp(secondary_debursted_crop)

## Interferometry

The cell below will only run if `get_complex` flag was set to True. In this case, the debursted crops will be used to create an interferogram here.

Also, the orbital (flat earth) phase component as well as the topographic phase component will be simulated and compensated.

It is also possible to compute the coherence. (`compute_coherence` flag)


In [ ]:
compute_coherence = True

In [ ]:
# Take assembler for products of primary date
primary_asm = mosaic.S1Assembler.from_products([products[0]], orbit_provider=orbit_provider, swaths=swaths)
primary_model = primary_asm.get_proj_model()

# do the same for the secondary date 
secondary_asm = mosaic.S1Assembler.from_products([products[1]], orbit_provider=orbit_provider, swaths=swaths)
secondary_model = secondary_asm.get_proj_model()

# Get the dem on the roi
dem = eos.dem.get_any_source()
margin = 500 # margin (padding) in pxs to add to the region of interest for dem download

refined_geom, _, _ = primary_model.get_approx_geom(roi_in_primary, margin)

dem_path = os.path.join(out_path, "dem.tif")
# get dem points
x, y, raster, transform, crs = eos.sar.regist.dem_points(refined_geom,
                                                         dem=dem,
                                                         outfile=dem_path)

In [ ]:
if get_complex:
    # if you wish to do the interferogram
    interf = primary_debursted_crop * np.conj(secondary_debursted_crop)
    write_array(interf, os.path.join(out_path, "interf.tif"))

    # create a TopoCorrection instance
    # This object computes the orbital and topographic phase
    # geometric quantities (baseline, incidence..)
    # are evaluated at a uniform grid (50x50 pixels in this case)
    # and a 2D polynomial of degree (7 here) is fitted for those quantities
    topo = eos.sar.geom_phase.TopoCorrection(primary_model, [secondary_model],
                                             grid_size=50, degree=7)

    # predict flat earth (orbital phase) and correction
    flat_earth = topo.flat_earth_image(roi_in_primary)
    flat_earth_interf_correction = np.exp(- 1j * flat_earth[0]).astype(np.complex64)
    del flat_earth
    flattened = interf * flat_earth_interf_correction
    write_array(flat_earth_interf_correction, os.path.join(out_path, "flat_earth.tif"))
    write_array(flattened, os.path.join(out_path, "flattened.tif"))
    del flat_earth_interf_correction

    # re-read the saved dem (downloaded before)
    with rasterio.open(dem_path, 'r') as dem_db:
        raster = dem_db.read(1)
        transform = dem_db.transform
        left, bottom, right, top = dem_db.bounds
        
    geometry = [(left, top), (right, top), (right, bottom), (left, bottom)]
    # Dem projection in radar coordinates
    heights = eos.sar.dem_to_radar.dem_radarcoding(raster, transform,
                                                   primary_model,
                                                   roi=roi_in_primary,
                                                   approx_geometry=geometry,
                                                   margin=margin)

    # predict topographic phase and correction
    topo_phase = topo.topo_phase_image(heights, primary_roi=roi_in_primary)

    topo_interf_correction = np.exp(- 1j * topo_phase[0]).astype(np.complex64)
    write_array(topo_interf_correction, os.path.join(out_path, "topo.tif"))
    del topo_phase

    corrected_interf = flattened * topo_interf_correction

    write_array(corrected_interf, os.path.join(out_path, "dinterf.tif"))

    if compute_coherence:
        filter_size = (5, 21)
        # It is also possible to compute the coherence
        coher = eos.sar.coherence.on_pair(
            primary_debursted_crop, secondary_debursted_crop,
            # here, you can modify the filter size if you wish
            filter_size=filter_size,
            might_contain_nans=True)
        write_array(coher, os.path.join(out_path, "coherence.tif"))

###  Plot Results

In [ ]:
filter_size = (5,21)
if get_complex:
    def display_interf(interf, filter_size=(5,21)):
        interf_disp = multilook(interf, filter_size)
        plt.figure(figsize=(12,10))
        plt.imshow(np.angle(interf_disp), cmap='jet', resample=False)
        plt.colorbar()
        plt.show()

    display_interf(interf, filter_size)
    display_interf(flattened, filter_size)
    display_interf(corrected_interf, filter_size)
    
    if compute_coherence:
        plt.figure(figsize=(12,10))
        plt.imshow(coher[::filter_size[0], ::filter_size[1]], cmap='gray', resample=False)
        plt.colorbar()
        plt.show()